# Dwarf Example 01: Your First Dwarf Rotation Curve

**EPS Research — Dwarf/Irregular HI Corpus v1.0**

Load and plot a dwarf irregular rotation curve from the corpus.
We use DDO154, one of the best-studied dark-matter-dominated dwarfs.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.20320362  
**Sources:** LVHIS (Koribalski 2019), VLA-ANGST (Ott 2012), LITTLE THINGS (Oh 2015), WALLABY DR2  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Colab setup: download corpus from Zenodo ──────────────────
import os, urllib.request
CORPORA = {
    'dwarf_irregular_corpus_v1.json': 'https://zenodo.org/records/20320362/files/dwarf_irregular_corpus_v1.json',
}
for filename, url in CORPORA.items():
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filename)
        print(f"  ✓ {filename}")
    else:
        print(f"  Already present: {filename}")
print("Ready.")


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('dwarf_irregular_corpus_v1.json') as f:
    corpus = json.load(f)

print(f"Total galaxies: {len(corpus['galaxies'])}")

# Find DDO154 (may be listed as DDO_154 or similar)
name = 'DDO154'
matches = [g for g in corpus['galaxies']
           if name.lower().replace(' ','') in g['galaxy'].lower().replace(' ','').replace('_','')]
if not matches:
    # Fall back to first LITTLE THINGS galaxy with data
    matches = [g for g in corpus['galaxies']
               if g.get('survey') == 'LITTLE_THINGS' and g.get('data')]
g = matches[0]

print(f"Galaxy:   {g['galaxy']}")
print(f"Survey:   {g['survey']}")
print(f"Distance: {g['distance_mpc']} Mpc")
print(f"Tier:     {g['quality_tier']}")
print(f"N rings:  {g.get('n_points', len(g.get('data', [])))}")


In [ ]:
d    = g.get('data', [])
if d:
    R    = [p['Rad']  for p in d]
    Vobs = [p['Vrot'] for p in d]
    errV = [p.get('errV', 0) for p in d]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(R, Vobs, yerr=errV, fmt='o-', color='#2ca02c',
                capsize=4, linewidth=1.8, markersize=6,
                label=r'$V_{\rm obs}$')
    ax.set_xlabel('Radius (kpc)', fontsize=12)
    ax.set_ylabel(r'$V_{\rm obs}$ (km/s)', fontsize=12)
    ax.set_title(f'{g["galaxy"]} — Dwarf Irregular Rotation Curve\n'
                 'EPS Research Dwarf/Irregular Corpus v1.0', fontsize=11)
    ax.legend()
    ax.text(0.97, 0.08, f'D={g["distance_mpc"]} Mpc\ninc={g["inc_deg"]}°',
            transform=ax.transAxes, ha='right', fontsize=9,
            bbox=dict(boxstyle='round', fc='white', alpha=0.7))
    plt.tight_layout()
    plt.savefig('dw01_dwarf_rc.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f"No per-ring data for {g['galaxy']} (Tier 2 — classification only)")
